# Research Paper Recommendation System  
## TF-IDF Based Recommendation Model

This notebook implements a TF-IDF based recommendation system for research papers.

The workflow includes:

- Loading cleaned dataset
- TF-IDF vectorization
- Similarity computation
- Nearest Neighbor search
- Research paper recommendation generation

TF-IDF acts as the baseline recommendation model before semantic search using BERT and FAISS.

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load cleaned dataset
import pandas as pd
df = pd.read_csv("../Dataset/cleaned_data.csv")

In [3]:
# Dataset Overview

#The cleaned dataset generated during preprocessing is loaded for TF-IDF vectorization.
print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (5000, 14)


,id,title,category,category_code,published_date,updated_date,authors,first_author,summary,summary_word_count,combined_text,clean_text,char_count,word_count
0,abs-1302.3557v1,Approximations for Decision Making in the Demp...,Artificial Intelligence,cs.AI,2/13/13,2/13/13,['Mathias Bauer'],'Mathias Bauer',The computational complexity of reasoning with...,98,Approximations for Decision Making in the Demp...,approximation decision making dempstershafer t...,544,62
1,abs-1509.03247v1,An Epsilon Hierarchical Fuzzy Twin Support Vec...,Artificial Intelligence,cs.AI,9/10/15,9/10/15,['Arindam Chaudhuri'],'Arindam Chaudhuri',The research presents epsilon hierarchical fuz...,129,An Epsilon Hierarchical Fuzzy Twin Support Vec...,epsilon hierarchical fuzzy twin support vector...,881,104
2,abs-1603.02738v1,Learning to Blend Computer Game Levels,Artificial Intelligence,cs.AI,3/8/16,3/8/16,"['Matthew Guzdial', 'Mark Riedl']",'Matthew Guzdial',We present an approach to generate novel compu...,122,Learning to Blend Computer Game Levels We pres...,learning blend computer game level present app...,651,83
3,abs-1203.3493v1,Solving Hybrid Influence Diagrams with Determi...,Artificial Intelligence,cs.AI,3/15/12,3/15/12,"['Yijing Li', 'Prakash P. Shenoy']",'Yijing Li',We describe a framework and an algorithm for s...,93,Solving Hybrid Influence Diagrams with Determi...,solving hybrid influence diagram deterministic...,604,65
4,abs-1106.0667v1,Reasoning within Fuzzy Description Logics,Artificial Intelligence,cs.AI,6/3/11,6/3/11,['U. Straccia'],'U. Straccia',"Description Logics (DLs) are suitable, well-kn...",133,Reasoning within Fuzzy Description Logics Desc...,reasoning within fuzzy description logic descr...,682,83


# TF-IDF Vectorization

TF-IDF converts textual data into numerical vectors based on word importance.

This serves as the baseline recommendation model.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

# Transform text
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (5000, 5000)


# Cosine Similarity

Cosine similarity measures similarity between research papers.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity matrix
from sklearn.neighbors import NearestNeighbors

# Build nearest neighbors model
nn_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=5
)

# Train model
nn_model.fit(tfidf_matrix)

print("NearestNeighbors model trained successfully!")

NearestNeighbors model trained successfully!


# Recommendation System

This function returns top similar research papers based on cosine similarity.

In [6]:
def recommend_papers(title, top_n=5):

    # Get paper index
    idx = df[df['title'] == title].index[0]

    # Find nearest neighbors
    distances, indices = nn_model.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=top_n + 1
    )

    # Extract recommended indices
    paper_indices = indices.flatten()[1:]

    # Return recommendations
    return df[['title', 'category']].iloc[paper_indices]

In [7]:
print(df['title'].iloc[0])

Approximations for Decision Making in the Dempster-Shafer Theory of
  Evidence


In [8]:
# Test recommendations
recommend_papers(df['title'].iloc[0])

,title,category
726,A New Technique for Combining Multiple Classif...,Artificial Intelligence
2413,D numbers theory: a generalization of Dempster...,Artificial Intelligence
30,On the Combinality of Evidence in the Dempster...,Artificial Intelligence
1062,Evidence as Opinions of Experts,Artificial Intelligence
4381,"Confidence Factors, Empiricism and the Dempste...",Artificial Intelligence


# Conclusion

In this notebook:

- TF-IDF vectorization was applied to research paper text
- Nearest Neighbor search was implemented
- A recommendation system was developed using cosine similarity

This TF-IDF model serves as the baseline recommendation engine before implementing semantic recommendation using BERT embeddings and FAISS.